# English Datasets Connectivity Test

Tests graph connectivity when using **strict** datasets for start/end words but **non-strict** datasets for intermediate steps (per docs/context.md policy).

- **Strict**: english_4_strict.txt, english_5_strict.txt (evaluation endpoints)
- **Non-strict**: english_4.txt, english_5.txt (allowed steps during search)

We build the graph on non-strict words and measure what fraction of strict words are reachable from each other.

## Metric explanations

| Metric | Meaning |
|--------|---------|
| **Non-strict nodes** | Total words in the graph. Each word is a node; you can step through any of these during a ladder. |
| **Edges** | Pairs of words that differ by exactly one letter. An edge means you can move directly between those two words in one step. |
| **Connected components** | Groups of words that can reach each other via one-letter steps. Words in different components have no path between them. |
| **Largest component** | The biggest such group. If most words are here, the graph is well-connected. |
| **Strict words in largest component** | How many of your evaluation endpoints (strict words) can reach each other when steps use non-strict. This is the key metric for ladder viability. |
| **Avg degree** | Average number of one-letter neighbors per word. Higher = more options at each step, easier to find paths. |

In [3]:
import networkx as nx
from pathlib import Path
import string

try:
    from tqdm import tqdm
except ImportError:
    tqdm = lambda x, **kw: x

BASE = Path("../data")
ENGLISH_LETTERS = string.ascii_lowercase


def load_words(path: Path) -> set[str]:
    return {w for w in path.read_text(encoding="utf-8").splitlines() if w.strip()}


def one_letter_neighbors(w: str, words: set[str], letters: str) -> set[str]:
    """Generate all words that differ from w by exactly one letter."""
    neighbors = set()
    for i in range(len(w)):
        for c in letters:
            if c != w[i]:
                candidate = w[:i] + c + w[i + 1:]
                if candidate in words:
                    neighbors.add(candidate)
    return neighbors


def connectivity_check(n: int):
    """
    Build graph on non-strict words. Report connectivity for strict words.
    - Nodes: non-strict (allowed steps)
    - Metric: what fraction of strict words are in the largest component?
    """
    strict_path = BASE / f"english_{n}_strict.txt"
    nonstrict_path = BASE / f"english_{n}.txt"
    if not strict_path.exists() or not nonstrict_path.exists():
        print(f"Missing data files for {n}-letter. Run 01_word_lists_combine.ipynb first.")
        return

    strict = load_words(strict_path)
    nonstrict = load_words(nonstrict_path)
    strict_in_graph = strict & nonstrict
    if len(strict_in_graph) < len(strict):
        print(f"Note: {len(strict) - len(strict_in_graph)} strict words not in non-strict (excluded from graph)")

    # Build graph on non-strict words
    G = nx.Graph()
    G.add_nodes_from(nonstrict)
    for w in tqdm(nonstrict, desc=f"{n}-letter edges"):
        for neighbor in one_letter_neighbors(w, nonstrict, ENGLISH_LETTERS):
            if w < neighbor:
                G.add_edge(w, neighbor)

    # Graph stats
    print(f"\n=== {n}-letter (strict endpoints, non-strict steps) ===")
    print(f"Non-strict nodes: {G.number_of_nodes()}")
    print(f"Edges: {G.number_of_edges()}")
    print(f"Connected components: {nx.number_connected_components(G)}")

    largest_cc = max(nx.connected_components(G), key=len)
    pct_all = len(largest_cc) / len(nonstrict) if nonstrict else 0
    print(f"Largest component: {len(largest_cc)} / {len(nonstrict)} ({pct_all:.1%}) of non-strict")

    # Strict words in largest component (only count strict words that are in the graph)
    strict_in_largest = strict_in_graph & largest_cc
    pct_strict = len(strict_in_largest) / len(strict_in_graph) if strict_in_graph else 0
    print(f"Strict words in largest component: {len(strict_in_largest)} / {len(strict_in_graph)} ({pct_strict:.1%})")

    degrees = [d for _, d in G.degree()]
    avg_deg = sum(degrees) / len(degrees) if degrees else 0
    print(f"Avg degree: {avg_deg:.2f}")

    if pct_strict > 0.95:
        print("→ Strict endpoints well-connected via non-strict steps.")
    else:
        isolated_strict = len(strict_in_graph) - len(strict_in_largest)
        print(f"→ {isolated_strict} strict words outside largest component.")


# Run for both 4-letter and 5-letter
connectivity_check(4)
connectivity_check(5)

4-letter edges: 100%|██████████| 5733/5733 [00:00<00:00, 26177.04it/s]



=== 4-letter (strict endpoints, non-strict steps) ===
Non-strict nodes: 5733
Edges: 38070
Connected components: 76
Largest component: 5643 / 5733 (98.4%) of non-strict
Strict words in largest component: 3155 / 3176 (99.3%)
Avg degree: 13.28
→ Strict endpoints well-connected via non-strict steps.
Note: 1 strict words not in non-strict (excluded from graph)


5-letter edges: 100%|██████████| 11155/11155 [00:00<00:00, 32439.07it/s]


=== 5-letter (strict endpoints, non-strict steps) ===
Non-strict nodes: 11155
Edges: 34531
Connected components: 967
Largest component: 9902 / 11155 (88.8%) of non-strict
Strict words in largest component: 5330 / 5810 (91.7%)
Avg degree: 6.19
→ 480 strict words outside largest component.


In [4]:
# Islands (components) with more than 10 words
# Run the cell above first so load_words, one_letter_neighbors, BASE, ENGLISH_LETTERS exist

def list_large_islands(n: int, min_size: int = 11):
    """List connected components with more than min_size words."""
    nonstrict_path = BASE / f"english_{n}.txt"
    if not nonstrict_path.exists():
        print(f"Missing {nonstrict_path}. Run 01_word_lists_combine.ipynb first.")
        return
    nonstrict = load_words(nonstrict_path)
    G = nx.Graph()
    G.add_nodes_from(nonstrict)
    for w in tqdm(nonstrict, desc=f"{n}-letter graph"):
        for neighbor in one_letter_neighbors(w, nonstrict, ENGLISH_LETTERS):
            if w < neighbor:
                G.add_edge(w, neighbor)

    components = list(nx.connected_components(G))
    large = [(cc, len(cc)) for cc in components if len(cc) > min_size]
    large.sort(key=lambda x: -x[1])

    total_nodes = len(nonstrict)
    largest_size = max(len(cc) for cc in components)
    smaller_total = total_nodes - largest_size

    print(f"\n=== {n}-letter islands (> {min_size} words) ===\n")
    for cc, size in large:
        words_sorted = sorted(cc)
        if size < 20:
            display = ', '.join(words_sorted)
        else:
            display = ', '.join(words_sorted[:3]) + ', ...'
        print(f"{size} words ({display})")
    print(f"\nWords in smaller islands combined: {smaller_total}")


list_large_islands(4)
list_large_islands(5)

4-letter graph: 100%|██████████| 5733/5733 [00:00<00:00, 32687.78it/s]



=== 4-letter islands (> 11 words) ===

5643 words (aahs, aals, abac, ...)

Words in smaller islands combined: 90


5-letter graph: 100%|██████████| 11155/11155 [00:00<00:00, 26585.02it/s]


=== 5-letter islands (> 11 words) ===

9902 words (aahed, abaca, abaci, ...)

Words in smaller islands combined: 1253


## Filter to largest component only

For gameplay, start and end words must be in the same island or no path exists. Smaller islands are too limited to be useful. The cell below writes to `data/islands/`:

| File | Steps | Endpoints |
|------|-------|----------|
| `english_n_largest_island.txt` | non-strict | non-strict |
| `english_n_strict_largest_island.txt` | non-strict | strict (in largest) |
| `english_n_strict_only_island.txt` | strict only | strict only |

In [5]:
ISLANDS_DIR = BASE / "islands"
ISLANDS_DIR.mkdir(exist_ok=True)


def filter_to_largest_component(n: int):
    """Keep only words in the largest component. Saves to data/islands/."""
    nonstrict_path = BASE / f"english_{n}.txt"
    strict_path = BASE / f"english_{n}_strict.txt"
    if not nonstrict_path.exists():
        print(f"Missing {nonstrict_path}. Run 01_word_lists_combine.ipynb first.")
        return
    nonstrict = load_words(nonstrict_path)
    strict = load_words(strict_path) if strict_path.exists() else set()

    # Graph on non-strict → largest island (steps can use non-strict)
    G = nx.Graph()
    G.add_nodes_from(nonstrict)
    for w in tqdm(nonstrict, desc=f"{n}-letter graph"):
        for neighbor in one_letter_neighbors(w, nonstrict, ENGLISH_LETTERS):
            if w < neighbor:
                G.add_edge(w, neighbor)

    largest = max(nx.connected_components(G), key=len)
    nonstrict_largest = sorted(nonstrict & largest)
    strict_largest = sorted(strict & largest)

    # Graph on strict only → largest island (steps restricted to strict)
    G_strict = nx.Graph()
    G_strict.add_nodes_from(strict)
    for w in tqdm(strict, desc=f"{n}-letter strict graph"):
        for neighbor in one_letter_neighbors(w, strict, ENGLISH_LETTERS):
            if w < neighbor:
                G_strict.add_edge(w, neighbor)
    largest_strict = max(nx.connected_components(G_strict), key=len)
    strict_only_island = sorted(largest_strict)

    out_largest = ISLANDS_DIR / f"english_{n}_largest_island.txt"
    out_strict = ISLANDS_DIR / f"english_{n}_strict_largest_island.txt"
    out_strict_only = ISLANDS_DIR / f"english_{n}_strict_only_island.txt"

    out_largest.write_text("\n".join(nonstrict_largest) + "\n", encoding="utf-8")
    out_strict.write_text("\n".join(strict_largest) + "\n", encoding="utf-8")
    out_strict_only.write_text("\n".join(strict_only_island) + "\n", encoding="utf-8")

    print(f"{n}-letter:")
    print(f"  largest_island: {len(nonstrict)} → {len(nonstrict_largest)} (non-strict steps)")
    print(f"  strict_largest_island: {len(strict)} → {len(strict_largest)} (strict endpoints, non-strict steps)")
    print(f"  strict_only_island: {len(strict)} → {len(strict_only_island)} (strict steps only)")
    print(f"  Saved to {ISLANDS_DIR}/")


filter_to_largest_component(4)
filter_to_largest_component(5)

4-letter strict graph: 100%|██████████| 3176/3176 [00:00<00:00, 36913.46it/s]


4-letter:
  largest_island: 5733 → 5643 (non-strict steps)
  strict_largest_island: 3176 → 3155 (strict endpoints, non-strict steps)
  strict_only_island: 3176 → 3050 (strict steps only)
  Saved to ..\data\islands/


5-letter strict graph: 100%|██████████| 5811/5811 [00:00<00:00, 32453.03it/s]

5-letter:
  largest_island: 11155 → 9902 (non-strict steps)
  strict_largest_island: 5811 → 5330 (strict endpoints, non-strict steps)
  strict_only_island: 5811 → 4573 (strict steps only)
  Saved to ..\data\islands/
